# Phase 4: Causal Patching — Activation Patching

**Experimental protocol (Denoising):**
1. Run **clean prompt** → save clean hidden states for all layers (at the last token of `<hop1>`)
2. Run **counterfactual (corrupted) prompt** normally → save the baseline (hallucinated) answer
3. For **each layer**: Run **counterfactual prompt again**, but REPLACE the layer's hidden state (at the last token of `<hop1>`) with the CLEAN state harvested in step 1
4. Measure whether injecting the clean state successfully restores the correct answer

If the correct answer is restored in ≥50% of cases for a given layer, that layer is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import torch
import numpy as np
import random

# Set seed for determinism
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
# ===== CELL 6 — extract hop-1 probe features under causal patching (needs the model, NOT the probes) =====
# Needs cells 1-3 (model, tokenizer, pairs). Saves everything to probe_features.npz so any probe
# (fixed, retrained, other layer) can be scored later in CELL 7 in seconds, without the GPU.
#
# Design: the hop-1 TEXT is held fixed (the clean run's correct hop 1) in every condition.
#   clean prompt     + clean hop 1  -> hop is grounded
#   corrupted prompt + same hop 1   -> hop doesn't fit the question
#   corrupted + entity patch at L   -> internal state restored
# Features are built exactly as in training (generate_cot.py): layer-L output, mean over the
# hop-1 inner-text tokens, default tokenizer special tokens.

import re, difflib, json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

FEATURE_LAYERS = [9, 10, 11, 12, 13]     # the whole probe plateau; probes at any of these can be scored later
MIN_GAP = 1.0                            # same validity filter as the causal-tracing cell
BATCH = 3                                # lower to 2 on OOM
CONTROL_LAYERS = list(range(0, 28, 3))   # shuffled-donor control at these layers
num_layers = model.config.num_hidden_layers
DEV = model.device
HOP1_RE = re.compile(r"<hop1>(.*?)</hop1>", re.DOTALL | re.IGNORECASE)

def build_inputs(prompt, cot_prefix, c0, c1):
    """Mirror generate_cot.extract_hidden_states tokenization + hop-1 span."""
    full = tokenizer(prompt + cot_prefix, return_tensors="pt").input_ids[0]
    p_len = len(tokenizer(prompt, return_tensors="pt").input_ids[0])
    enc = tokenizer(cot_prefix, return_offsets_mapping=True, add_special_tokens=False)
    if len(full) != p_len + len(enc["input_ids"]):
        return None
    offs = enc["offset_mapping"]
    ts = next((i for i, (cs, ce) in enumerate(offs) if cs <= c0 < ce), None)
    te = next((i for i, (cs, ce) in enumerate(offs) if cs < c1 <= ce), None)
    if ts is None or te is None:
        return None
    return full, p_len, (p_len + ts, p_len + te + 1)

def make_store_hook(store, key):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        store[key] = h[0].detach().clone()
        return output
    return hook

@torch.no_grad()
def run_rows(input_ids, P, mask, span, configs):
    """Each config = list of (module, positions, vectors) patches for one batch row.
    Returns [(masked target log-prob, array[len(FEATURE_LAYERS), d] pooled hop-1 features), ...]."""
    out = []
    for s in range(0, len(configs), BATCH):
        chunk = configs[s:s + BATCH]
        by_mod = {}
        for b, cfg in enumerate(chunk):
            for mod, pos, vecs in cfg:
                by_mod.setdefault(mod, []).append((b, pos, vecs))
        handles, cap = [], {}
        for mod, items in by_mod.items():                       # patch hooks first ...
            def hook(module, args, output, items=items):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone()
                for b, pos, vecs in items:
                    h[b, pos.to(h.device), :] = vecs.to(h.device)
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handles.append(mod.register_forward_hook(hook))
        for Pl in FEATURE_LAYERS:                                # ... then capture (sees patched output)
            def cap_hook(module, args, output, Pl=Pl):
                h = output[0] if isinstance(output, tuple) else output
                cap[Pl] = h[:, span[0]:span[1], :].float().mean(1).cpu().numpy()
            handles.append(model.model.layers[Pl].register_forward_hook(cap_hook))
        try:
            ids = input_ids.unsqueeze(0).repeat(len(chunk), 1).to(DEV)
            hid = model.model(input_ids=ids, use_cache=False).last_hidden_state[:, P - 1:-1]
            lp = F.log_softmax(model.lm_head(hid).float(), dim=-1)
        finally:
            for hd in handles:
                hd.remove()
        tgt = input_ids[P:].to(lp.device)
        lp = lp.gather(2, tgt.view(1, -1, 1).expand(len(chunk), -1, 1)).squeeze(2)
        lps = lp[:, mask.to(lp.device)].sum(1).tolist()
        for b in range(len(chunk)):
            out.append((lps[b], np.stack([cap[Pl][b] for Pl in FEATURE_LAYERS])))
    return out

ids, gaps, recov, F_clean, F_cf, F_entity, F_control, has_control = [], [], [], [], [], [], [], []
skipped = {"no_hop1": 0, "tok_mismatch": 0, "no_diff": 0, "no_gap": 0}
donor = None   # previous pair's clean entity vectors -> shuffled control
d_model = model.config.hidden_size

for idx, (clean, cf) in enumerate(tqdm(pairs, desc="Extracting probe features")):
    cot = clean["generated_cot"]
    m = HOP1_RE.search(cot)
    if not m:
        skipped["no_hop1"] += 1; continue
    cot_prefix = cot[:m.end()]
    cf_prompt = (clean["prompt"].replace(clean["question"], cf["question"])
                 if clean["question"] in clean["prompt"] else cf["prompt"])
    bc = build_inputs(clean["prompt"], cot_prefix, m.start(1), m.end(1))
    bf = build_inputs(cf_prompt, cot_prefix, m.start(1), m.end(1))
    if bc is None or bf is None:
        skipped["tok_mismatch"] += 1; continue
    c_in, Pc, span_c = bc
    f_in, Pf, span_f = bf

    sm = difflib.SequenceMatcher(a=c_in[:Pc].tolist(), b=f_in[:Pf].tolist(), autojunk=False)
    bl = [(i1, i2, j1, j2) for t, i1, i2, j1, j2 in sm.get_opcodes() if t != "equal"]
    if not bl:
        skipped["no_diff"] += 1; continue
    i1, i2 = min(b[0] for b in bl), max(b[1] for b in bl)
    j1, j2 = min(b[2] for b in bl), max(b[3] for b in bl)
    if i2 == i1 or j2 == j1:
        skipped["no_diff"] += 1; continue
    c_pos = torch.tensor([i2 - 1]); f_pos = torch.tensor([j2 - 1])

    t_ids = c_in[Pc:]
    mask = ~torch.isin(t_ids, torch.cat([c_in[i1:i2], f_in[j1:j2]]))
    if mask.sum() == 0:
        skipped["no_gap"] += 1; continue

    store = {}
    hs = [model.model.layers[l].register_forward_hook(make_store_hook(store, l)) for l in range(num_layers)]
    try:
        (lp_clean, fc), = run_rows(c_in, Pc, mask, span_c, [[]])
    finally:
        for h in hs:
            h.remove()
    (lp_cf, ff), = run_rows(f_in, Pf, mask, span_f, [[]])
    gap = lp_clean - lp_cf
    if gap < MIN_GAP:
        skipped["no_gap"] += 1; continue

    Ls = model.model.layers
    configs = [[(Ls[l], f_pos, store[l][c_pos.to(store[l].device)])] for l in range(num_layers)]
    if donor is not None:
        configs += [[(Ls[l], f_pos, donor[l])] for l in CONTROL_LAYERS]
    res = run_rows(f_in, Pf, mask, span_f, configs)

    ids.append(str(clean.get("id"))); gaps.append(gap)
    recov.append([(res[l][0] - lp_cf) / gap for l in range(num_layers)])
    F_clean.append(fc.astype(np.float16)); F_cf.append(ff.astype(np.float16))
    F_entity.append(np.stack([res[l][1] for l in range(num_layers)]).astype(np.float16))
    if donor is not None:
        F_control.append(np.stack([r[1] for r in res[num_layers:]]).astype(np.float16)); has_control.append(True)
    else:
        F_control.append(np.full((len(CONTROL_LAYERS), len(FEATURE_LAYERS), d_model), np.nan, np.float16)); has_control.append(False)

    donor = {l: store[l][c_pos.to(store[l].device)].clone() for l in range(num_layers)}
    del store
    torch.cuda.empty_cache()

n = len(ids)
print(f"\nPairs: {len(pairs)} | valid: {n} | skipped: {skipped}")
assert n > 0
np.savez("probe_features.npz",
         ids=np.array(ids), gap=np.array(gaps, np.float32), recovery=np.array(recov, np.float32),
         F_clean=np.stack(F_clean), F_cf=np.stack(F_cf), F_entity=np.stack(F_entity),
         F_control=np.stack(F_control), has_control=np.array(has_control),
         feature_layers=np.array(FEATURE_LAYERS), control_layers=np.array(CONTROL_LAYERS))
json.dump(skipped, open("probe_features_skipped.json", "w"))
print("Saved probe_features.npz — now run CELL 7 (scoring) whenever the probes are ready.")
